# Two-Stage ONNX Ensemble Production Pipeline

## Architecture
- **Stage 1**: 3 Binary ensemble models (Cancer vs Non-cancer) using ConservativeMobileNetV3
- **Stage 2**: 3 Four-class ensemble models (Benign, Healthy, OCA, OPMD) using ConservativeMobileNetV3
- **Unknown**: Handled via confidence thresholding at inference time

## Baseline to Beat
- **ONNX 5-class ensemble**: 76.36% accuracy

## Training Setup
Matches `production_oral_cavity_classifier.ipynb` exactly:
- ConservativeMobileNetV3 (MobileNetV3Large backbone)
- 3 model variants with different dropout rates (0.3, 0.1, 0.2)
- Adam optimizer, lr=1e-4
- StepLR scheduler (step=15, gamma=0.1)
- Weighted ensemble voting (0.30, 0.40, 0.30)

In [ ]:
# Cell 1: Config and Setup

import os
import json
import random
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

# Configuration - matches production_oral_cavity_classifier.ipynb
CONFIG = {
    # Data paths
    'data_dir_binary': os.path.expanduser('~/dev/oral_cavity_binary_dataset'),
    'data_dir_5class': os.path.expanduser('~/dev/oral_cavity_5class_dataset'),
    'ensemble_test_dir': os.path.expanduser('~/dev/oral_cavity_5class_dataset/test'),
    
    # Model config - EXACT match to best ensemble model
    'image_size': (224, 224),
    'batch_size': 32,
    'learning_rate': 1e-4,
    'epochs': 30,
    'early_stopping_patience': 5,
    
    # Ensemble config
    'dropout_rates': [0.3, 0.1, 0.2],  # 3 variants
    'ensemble_weights': [0.30, 0.40, 0.30],
    
    # LR scheduler (matches PyTorch ensemble)
    'scheduler_step': 15,
    'scheduler_gamma': 0.1,
    
    # Class names
    'class_names_5class': ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown'],
    'class_names_4class': ['Benign', 'Healthy', 'OCA', 'OPMD'],
    'class_names_binary': ['non-cancer', 'cancer'],
    
    # Inference thresholds
    'cancer_threshold': 0.5,
    'unknown_threshold': 0.55,
}

# Output directory
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path(f'results/two_stage_onnx_ensemble/run_{timestamp}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(exist_ok=True)
(OUTPUT_DIR / 'onnx').mkdir(exist_ok=True)
(OUTPUT_DIR / 'plots').mkdir(exist_ok=True)

# Save config
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"\nConfiguration:")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Device: {DEVICE}")
print(f"  Image size: {CONFIG['image_size']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Epochs: {CONFIG['epochs']}")
print(f"  Ensemble dropout rates: {CONFIG['dropout_rates']}")
print(f"  Ensemble weights: {CONFIG['ensemble_weights']}")

In [ ]:
# Cell 2: Data Loading with PyTorch DataLoaders

# Transforms - EXACT match to best ensemble model
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['image_size'][0], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'][0], CONFIG['image_size'][1])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class OralCavityDataset(Dataset):
    """Generic dataset for oral cavity classification."""
    
    def __init__(self, image_paths: List[str], labels: List[int], transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        
        return img, label

def load_dataset_paths(base_dir: str, split: str, class_names: List[str]) -> Tuple[List[str], List[int]]:
    """Load image paths and labels for a dataset split."""
    paths = []
    labels = []
    
    split_dir = Path(base_dir) / split
    for class_idx, class_name in enumerate(class_names):
        class_dir = split_dir / class_name
        if class_dir.exists():
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in class_dir.glob(ext):
                    paths.append(str(img_path))
                    labels.append(class_idx)
    
    return paths, labels

def load_binary_dataset(base_dir: str) -> Tuple[List[str], List[int]]:
    """Load binary dataset (non-cancer=0, cancer=1)."""
    paths = []
    labels = []
    
    base_path = Path(base_dir)
    for folder, label in [('non-cancer', 0), ('cancer', 1)]:
        folder_path = base_path / folder
        if folder_path.exists():
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in folder_path.glob(ext):
                    paths.append(str(img_path))
                    labels.append(label)
    
    return paths, labels

print("Loading datasets...")
print("=" * 60)

# --- BINARY DATASET ---
print("\n[1] Binary Dataset (Cancer vs Non-cancer)")
binary_paths, binary_labels = load_binary_dataset(CONFIG['data_dir_binary'])
print(f"    Total samples: {len(binary_paths)}")
print(f"    Cancer: {sum(binary_labels)}, Non-cancer: {len(binary_labels) - sum(binary_labels)}")

# Split binary into train/val (80/20)
from sklearn.model_selection import train_test_split
binary_train_paths, binary_val_paths, binary_train_labels, binary_val_labels = train_test_split(
    binary_paths, binary_labels, test_size=0.2, random_state=SEED, stratify=binary_labels
)
print(f"    Train: {len(binary_train_paths)}, Val: {len(binary_val_paths)}")

# --- 4-CLASS DATASET (excluding Unknown) ---
print("\n[2] 4-Class Dataset (excluding Unknown)")
class_names_4 = CONFIG['class_names_4class']

train_4class_paths, train_4class_labels = load_dataset_paths(
    CONFIG['data_dir_5class'], 'train', class_names_4
)
val_4class_paths, val_4class_labels = load_dataset_paths(
    CONFIG['data_dir_5class'], 'val', class_names_4
)
test_4class_paths, test_4class_labels = load_dataset_paths(
    CONFIG['data_dir_5class'], 'test', class_names_4
)

print(f"    Train: {len(train_4class_paths)}")
print(f"    Val: {len(val_4class_paths)}")
print(f"    Test: {len(test_4class_paths)}")

# --- 5-CLASS TEST SET (for final evaluation) ---
print("\n[3] 5-Class Test Set (for evaluation)")
test_5class_paths, test_5class_labels = load_dataset_paths(
    CONFIG['data_dir_5class'], 'test', CONFIG['class_names_5class']
)
print(f"    Test: {len(test_5class_paths)}")
for i, name in enumerate(CONFIG['class_names_5class']):
    count = sum(1 for l in test_5class_labels if l == i)
    print(f"        {name}: {count}")

# Create DataLoaders
print("\n[4] Creating DataLoaders...")

# Binary datasets
train_ds_binary = OralCavityDataset(binary_train_paths, binary_train_labels, train_transforms)
val_ds_binary = OralCavityDataset(binary_val_paths, binary_val_labels, val_test_transforms)

train_loader_binary = DataLoader(train_ds_binary, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
val_loader_binary = DataLoader(val_ds_binary, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

# 4-class datasets
train_ds_4class = OralCavityDataset(train_4class_paths, train_4class_labels, train_transforms)
val_ds_4class = OralCavityDataset(val_4class_paths, val_4class_labels, val_test_transforms)

train_loader_4class = DataLoader(train_ds_4class, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
val_loader_4class = DataLoader(val_ds_4class, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

print("Datasets ready!")

In [ ]:
# Cell 3: Model Architecture - ConservativeMobileNetV3 (matches best ensemble)

class ConservativeMobileNetV3(nn.Module):
    """
    Conservative MobileNetV3Large architecture - proven to work.
    Exact match to production_oral_cavity_classifier.ipynb.
    """
    
    def __init__(self, num_classes: int, dropout_rate: float = 0.2):
        super(ConservativeMobileNetV3, self).__init__()
        
        # Load pre-trained MobileNetV3Large
        self.backbone = models.mobilenet_v3_large(pretrained=True)
        
        # Remove the original classifier
        self.backbone.classifier = nn.Identity()
        
        # Conservative classifier (simple and stable)
        # MobileNetV3Large features output: 960
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(960, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x

# Binary model variants (for ensemble)
class BinaryModel1(ConservativeMobileNetV3):
    def __init__(self):
        super().__init__(num_classes=1, dropout_rate=0.3)

class BinaryModel2(ConservativeMobileNetV3):
    def __init__(self):
        super().__init__(num_classes=1, dropout_rate=0.1)

class BinaryModel3(ConservativeMobileNetV3):
    def __init__(self):
        super().__init__(num_classes=1, dropout_rate=0.2)

# 4-class model variants (for ensemble)
class FourClassModel1(ConservativeMobileNetV3):
    def __init__(self):
        super().__init__(num_classes=4, dropout_rate=0.3)

class FourClassModel2(ConservativeMobileNetV3):
    def __init__(self):
        super().__init__(num_classes=4, dropout_rate=0.1)

class FourClassModel3(ConservativeMobileNetV3):
    def __init__(self):
        super().__init__(num_classes=4, dropout_rate=0.2)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Architecture: ConservativeMobileNetV3")
print("=" * 60)

# Test model creation
test_model = ConservativeMobileNetV3(num_classes=4, dropout_rate=0.2).to(DEVICE)
print(f"Parameters: {count_parameters(test_model):,}")
print(f"Model size: {count_parameters(test_model) * 4 / (1024*1024):.1f} MB")

# Test forward pass
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    output = test_model(dummy_input)
print(f"Output shape: {output.shape}")

del test_model
print("\nModel architecture verified!")

In [ ]:
# Cell 4: Training Function

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    num_epochs: int,
    patience: int,
    device: torch.device,
    model_name: str,
    is_binary: bool = False
) -> Dict:
    """
    Train a model with early stopping.
    
    Returns dict with training history and best model state.
    """
    model = model.to(device)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    print(f"\nTraining {model_name}...")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            if is_binary:
                labels = labels.float().unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            
            if is_binary:
                preds = (torch.sigmoid(outputs) > 0.5).float()
                train_correct += (preds == labels).sum().item()
            else:
                _, preds = torch.max(outputs, 1)
                train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                
                if is_binary:
                    labels = labels.float().unsqueeze(1)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                
                if is_binary:
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                    val_correct += (preds == labels).sum().item()
                else:
                    _, preds = torch.max(outputs, 1)
                    val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        # Step scheduler
        scheduler.step()
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Check for best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            save_marker = '*'
        else:
            patience_counter += 1
            save_marker = ''
        
        print(f"  Epoch {epoch+1}/{num_epochs}: "
              f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f} {save_marker}")
        
        # Early stopping
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # Load best model state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return {
        'history': history,
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'model': model
    }

print("Training function defined.")

In [ ]:
# Cell 5: Train Binary Ensemble Models (3 variants)

print("Training Binary Ensemble Models")
print("=" * 60)

binary_model_classes = [BinaryModel1, BinaryModel2, BinaryModel3]
binary_models = []
binary_results = []

for i, (model_class, dropout_rate) in enumerate(zip(binary_model_classes, CONFIG['dropout_rates'])):
    print(f"\n[Model {i+1}/3] Dropout rate: {dropout_rate}")
    print("-" * 40)
    
    model = model_class()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, 
        step_size=CONFIG['scheduler_step'], 
        gamma=CONFIG['scheduler_gamma']
    )
    
    result = train_model(
        model=model,
        train_loader=train_loader_binary,
        val_loader=val_loader_binary,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=CONFIG['epochs'],
        patience=CONFIG['early_stopping_patience'],
        device=DEVICE,
        model_name=f'binary_model_{i+1}',
        is_binary=True
    )
    
    # Save model
    model_path = OUTPUT_DIR / 'models' / f'binary_model_{i+1}.pth'
    torch.save(result['model'].state_dict(), model_path)
    print(f"  Saved: {model_path}")
    
    binary_models.append(result['model'])
    binary_results.append(result)

print("\nBinary ensemble training complete!")
for i, result in enumerate(binary_results):
    print(f"  Model {i+1}: Val Acc = {result['best_val_acc']:.4f}")

In [ ]:
# Cell 6: Train 4-Class Ensemble Models (3 variants)

print("Training 4-Class Ensemble Models")
print("=" * 60)

four_class_model_classes = [FourClassModel1, FourClassModel2, FourClassModel3]
four_class_models = []
four_class_results = []

for i, (model_class, dropout_rate) in enumerate(zip(four_class_model_classes, CONFIG['dropout_rates'])):
    print(f"\n[Model {i+1}/3] Dropout rate: {dropout_rate}")
    print("-" * 40)
    
    model = model_class()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, 
        step_size=CONFIG['scheduler_step'], 
        gamma=CONFIG['scheduler_gamma']
    )
    
    result = train_model(
        model=model,
        train_loader=train_loader_4class,
        val_loader=val_loader_4class,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=CONFIG['epochs'],
        patience=CONFIG['early_stopping_patience'],
        device=DEVICE,
        model_name=f'four_class_model_{i+1}',
        is_binary=False
    )
    
    # Save model
    model_path = OUTPUT_DIR / 'models' / f'four_class_model_{i+1}.pth'
    torch.save(result['model'].state_dict(), model_path)
    print(f"  Saved: {model_path}")
    
    four_class_models.append(result['model'])
    four_class_results.append(result)

print("\n4-Class ensemble training complete!")
for i, result in enumerate(four_class_results):
    print(f"  Model {i+1}: Val Acc = {result['best_val_acc']:.4f}")

In [ ]:
# Cell 7: ONNX Conversion

print("Converting models to ONNX...")
print("=" * 60)

def convert_to_onnx(model: nn.Module, output_path: str, input_shape=(1, 3, 224, 224)):
    """Convert PyTorch model to ONNX format."""
    model = model.to('cpu')
    model.eval()
    
    dummy_input = torch.randn(input_shape)
    
    try:
        torch.onnx.export(
            model,
            dummy_input,
            output_path,
            export_params=True,
            opset_version=12,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            }
        )
        
        # Verify ONNX model
        import onnx
        onnx_model = onnx.load(output_path)
        onnx.checker.check_model(onnx_model)
        
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"    Saved: {output_path} ({size_mb:.2f} MB)")
        return True, size_mb
    except Exception as e:
        print(f"    ERROR: {e}")
        return False, 0

total_size = 0

# Convert binary models
print("\n[1] Binary Models")
for i, model in enumerate(binary_models):
    output_path = str(OUTPUT_DIR / 'onnx' / f'binary_model_{i+1}.onnx')
    success, size = convert_to_onnx(model, output_path)
    if success:
        total_size += size

# Convert 4-class models
print("\n[2] 4-Class Models")
for i, model in enumerate(four_class_models):
    output_path = str(OUTPUT_DIR / 'onnx' / f'four_class_model_{i+1}.onnx')
    success, size = convert_to_onnx(model, output_path)
    if success:
        total_size += size

print(f"\nTotal ONNX size: {total_size:.2f} MB")
print("ONNX conversion complete!")

In [ ]:
# Cell 8: Two-Stage Ensemble Inference Logic

print("Setting up two-stage ensemble inference...")
print("=" * 60)

class TwoStageEnsembleClassifier:
    """
    Two-stage oral cavity classifier with ensemble voting.
    
    Stage 1: 3 Binary ensemble models predict cancer vs non-cancer
    Stage 2: 3 Four-class ensemble models predict Benign/Healthy/OCA/OPMD
    Unknown: Emitted when confidence is below threshold or models disagree
    """
    
    def __init__(self, binary_models: List[nn.Module], four_class_models: List[nn.Module],
                 ensemble_weights: List[float], cancer_threshold=0.5, unknown_threshold=0.55,
                 device=torch.device('cpu')):
        self.binary_models = [m.to(device).eval() for m in binary_models]
        self.four_class_models = [m.to(device).eval() for m in four_class_models]
        self.ensemble_weights = ensemble_weights
        self.cancer_threshold = cancer_threshold
        self.unknown_threshold = unknown_threshold
        self.device = device
        self.class_names = ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown']
        
        # Normalization (same as training)
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    
    def predict_batch(self, images: torch.Tensor) -> List[Dict]:
        """
        Predict classes for a batch of images.
        
        Args:
            images: Batch of preprocessed images (N, 3, 224, 224), normalized
        
        Returns:
            list of prediction dicts
        """
        images = images.to(self.device)
        
        # Stage 1: Binary ensemble
        binary_probs = []
        with torch.no_grad():
            for model in self.binary_models:
                outputs = torch.sigmoid(model(images))
                binary_probs.append(outputs.cpu().numpy().flatten())
        
        # Weighted average for binary
        binary_probs = np.array(binary_probs)  # (3, N)
        weights = np.array(self.ensemble_weights).reshape(-1, 1)
        cancer_probs = (binary_probs * weights).sum(axis=0)  # (N,)
        
        # Stage 2: 4-class ensemble
        four_class_probs = []
        with torch.no_grad():
            for model in self.four_class_models:
                outputs = torch.softmax(model(images), dim=1)
                four_class_probs.append(outputs.cpu().numpy())
        
        # Weighted average for 4-class
        four_class_probs = np.array(four_class_probs)  # (3, N, 4)
        weights_4 = np.array(self.ensemble_weights).reshape(-1, 1, 1)
        four_probs = (four_class_probs * weights_4).sum(axis=0)  # (N, 4)
        
        # Decision logic
        results = []
        for i in range(len(images)):
            cancer_prob = float(cancer_probs[i])
            probs_4 = four_probs[i]
            top_idx = int(np.argmax(probs_4))
            top_conf = float(np.max(probs_4))
            
            if top_conf < self.unknown_threshold:
                final_class = 4  # Unknown
                reason = 'low_confidence'
            elif cancer_prob >= self.cancer_threshold:
                if top_idx in [2, 3]:  # OCA or OPMD
                    final_class = top_idx
                    reason = 'cancer_match'
                else:
                    final_class = 3  # OPMD (conservative)
                    reason = 'cancer_override'
            else:
                if top_idx in [0, 1]:  # Benign or Healthy
                    final_class = top_idx
                    reason = 'non_cancer_match'
                else:
                    final_class = 4  # Unknown
                    reason = 'conflict'
            
            results.append({
                'final_class': final_class,
                'cancer_prob': cancer_prob,
                'four_class_conf': top_conf,
                'four_class_top': top_idx,
                'reason': reason
            })
        
        return results

# Create classifier instance
classifier = TwoStageEnsembleClassifier(
    binary_models=binary_models,
    four_class_models=four_class_models,
    ensemble_weights=CONFIG['ensemble_weights'],
    cancer_threshold=CONFIG['cancer_threshold'],
    unknown_threshold=CONFIG['unknown_threshold'],
    device=DEVICE
)

print(f"Two-stage ensemble classifier ready!")
print(f"  Binary models: {len(binary_models)}")
print(f"  4-class models: {len(four_class_models)}")
print(f"  Ensemble weights: {CONFIG['ensemble_weights']}")
print(f"  Cancer threshold: {CONFIG['cancer_threshold']}")
print(f"  Unknown threshold: {CONFIG['unknown_threshold']}")

In [ ]:
# Cell 9: Evaluation - Apples-to-Apples Comparison with Ensemble Baseline

print("Evaluating Two-Stage Ensemble Pipeline on 5-Class Test Set")
print("=" * 60)
print(f"Baseline to beat: ONNX 5-class ensemble = 76.36% accuracy")
print()

# Create test dataset
test_ds_5class = OralCavityDataset(test_5class_paths, test_5class_labels, val_test_transforms)
test_loader_5class = DataLoader(test_ds_5class, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

# Evaluate
all_preds = []
all_labels = []

print(f"Running inference...")
for images, labels in tqdm(test_loader_5class, desc='Evaluating'):
    results = classifier.predict_batch(images)
    all_preds.extend([r['final_class'] for r in results])
    all_labels.extend(labels.numpy().tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
overall_accuracy = float(np.mean(all_preds == all_labels))

print(f"\n" + "=" * 60)
print(f"RESULTS: Two-Stage ONNX Ensemble Pipeline")
print(f"=" * 60)
print(f"\nOverall Accuracy: {overall_accuracy*100:.2f}%")
print(f"Baseline (ONNX 5-class ensemble): 76.36%")
print(f"Difference: {(overall_accuracy*100 - 76.36):+.2f}%")

if overall_accuracy > 0.7636:
    print(f"\n>>> SUCCESS: Two-stage ensemble beats baseline! <<<")
else:
    print(f"\n>>> Two-stage ensemble did not beat baseline <<<")

# Per-class accuracy
print(f"\nPer-Class Accuracy:")
for i, name in enumerate(CONFIG['class_names_5class']):
    mask = all_labels == i
    if mask.sum() > 0:
        class_acc = float(np.mean(all_preds[mask] == all_labels[mask]))
        print(f"  {name}: {class_acc*100:.2f}% ({mask.sum()} samples)")
    else:
        print(f"  {name}: N/A (0 samples)")

# Confusion matrix
print(f"\nConfusion Matrix:")
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds, labels=range(5))
print(f"             Predicted")
print(f"             " + " ".join([f"{n[:4]:>6}" for n in CONFIG['class_names_5class']]))
for i, name in enumerate(CONFIG['class_names_5class']):
    row = " ".join([f"{cm[i,j]:>6}" for j in range(5)])
    print(f"  {name:>8}: {row}")

# Prediction distribution
print(f"\nPrediction Distribution:")
unique, counts = np.unique(all_preds, return_counts=True)
for u, c in zip(unique, counts):
    pct = c / len(all_preds) * 100
    print(f"  {CONFIG['class_names_5class'][u]}: {c} ({pct:.1f}%)")

# Save results
results_summary = {
    'overall_accuracy': overall_accuracy,
    'baseline_accuracy': 0.7636,
    'improvement': overall_accuracy - 0.7636,
    'total_samples': len(all_labels),
    'per_class_accuracy': {},
    'confusion_matrix': cm.tolist()
}
for i, name in enumerate(CONFIG['class_names_5class']):
    mask = all_labels == i
    if mask.sum() > 0:
        results_summary['per_class_accuracy'][name] = float(np.mean(all_preds[mask] == all_labels[mask]))

with open(OUTPUT_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nResults saved to: {OUTPUT_DIR / 'evaluation_results.json'}")

In [ ]:
# Cell 10: Summary and Deployment Instructions

print("=" * 60)
print("TWO-STAGE ONNX ENSEMBLE PIPELINE - SUMMARY")
print("=" * 60)

print(f"\nOutput Directory: {OUTPUT_DIR}")

print(f"\nBinary Ensemble Models (Cancer vs Non-cancer):")
for i in range(3):
    pth_path = OUTPUT_DIR / 'models' / f'binary_model_{i+1}.pth'
    onnx_path = OUTPUT_DIR / 'onnx' / f'binary_model_{i+1}.onnx'
    print(f"  [{i+1}] Dropout={CONFIG['dropout_rates'][i]}, Weight={CONFIG['ensemble_weights'][i]}")
    print(f"      PyTorch: {pth_path}")
    print(f"      ONNX: {onnx_path}")

print(f"\n4-Class Ensemble Models (Benign, Healthy, OCA, OPMD):")
for i in range(3):
    pth_path = OUTPUT_DIR / 'models' / f'four_class_model_{i+1}.pth'
    onnx_path = OUTPUT_DIR / 'onnx' / f'four_class_model_{i+1}.onnx'
    print(f"  [{i+1}] Dropout={CONFIG['dropout_rates'][i]}, Weight={CONFIG['ensemble_weights'][i]}")
    print(f"      PyTorch: {pth_path}")
    print(f"      ONNX: {onnx_path}")

# Calculate total size
total_onnx_size = 0
for i in range(3):
    binary_path = OUTPUT_DIR / 'onnx' / f'binary_model_{i+1}.onnx'
    four_class_path = OUTPUT_DIR / 'onnx' / f'four_class_model_{i+1}.onnx'
    if binary_path.exists():
        total_onnx_size += binary_path.stat().st_size
    if four_class_path.exists():
        total_onnx_size += four_class_path.stat().st_size
total_onnx_size_mb = total_onnx_size / (1024 * 1024)

print(f"\nTotal ONNX Size: {total_onnx_size_mb:.2f} MB")

print(f"\nInference Thresholds:")
print(f"  Cancer threshold: {CONFIG['cancer_threshold']}")
print(f"  Unknown threshold: {CONFIG['unknown_threshold']}")
print(f"  Ensemble weights: {CONFIG['ensemble_weights']}")

print(f"\n" + "=" * 60)
print("DEPLOYMENT INSTRUCTIONS")
print("=" * 60)

print(f"""
1. Copy ONNX models to Flutter assets:
   cp {OUTPUT_DIR / 'onnx' / '*.onnx'} ../../assets/

2. Update Flutter model_inference_service.dart:
   - Load all 6 ONNX models (3 binary + 3 four-class)
   - Implement weighted ensemble averaging
   - Implement two-stage inference logic

3. Ensemble Inference Flow:
   a) Run 3 binary models, compute weighted average:
      cancer_prob = sum(weight[i] * binary_model[i](image))
   
   b) Run 3 four-class models, compute weighted average:
      four_probs = sum(weight[i] * four_class_model[i](image))
   
   c) Apply decision logic:
      - If max(four_probs) < {CONFIG['unknown_threshold']}: return Unknown
      - If cancer_prob >= {CONFIG['cancer_threshold']}:
          - If four_probs says OCA/OPMD: return that class
          - Else: return OPMD (conservative)
      - Else (non-cancer):
          - If four_probs says Benign/Healthy: return that class
          - Else: return Unknown (conflict)

4. Ensemble Weights:
   Model 1: {CONFIG['ensemble_weights'][0]}
   Model 2: {CONFIG['ensemble_weights'][1]}
   Model 3: {CONFIG['ensemble_weights'][2]}
""")

print("\nTraining complete!")